# NC MNIST — Weight Decay Sweep

**The key experiment for the paper's central claim.**

Tests whether feature norm at T_NC stays ~1.06 as weight decay
varies — i.e., λ controls *speed* of collapse (T_NC), not the
*threshold* (fn at T_NC).

This is the direct analogue of the grokking paper's H4 finding.

| λ | Seeds | Phase2 budget | Expected |
|---|---|---|---|
| 1e-5 | 3 | 600 ep | Slow — may DNF |
| 5e-5 | 3 | 600 ep | Slower than baseline |
| 1e-4 | 1 | — | **Baseline** T_NC=310 fn=1.063 ✓ |
| 5e-4 | 3 | 600 ep | Faster than baseline |

**Outputs:** `sweep_wd.csv`, `fig_wd_sweep.png`, per-run CSVs `wd{value}_s{seed}.csv`

**Est. runtime: ~45 min on T4**

**Fully self-contained. Settings → T4 GPU → Save & Run All.**

In [1]:
import torch, torchvision, time
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = '/kaggle/working/'
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
print(f'GPU: {torch.cuda.get_device_name(0)}  |  PyTorch: {torch.__version__}')

# Baseline already confirmed: wd=1e-4, T_NC=310, fn=1.063
BASELINE_WD  = 1e-4
BASELINE_TNC = 310
BASELINE_FN  = 1.063
print(f'Baseline: wd={BASELINE_WD}  T_NC={BASELINE_TNC}  fn={BASELINE_FN}')


GPU: Tesla T4  |  PyTorch: 2.10.0+cu128
Baseline: wd=0.0001  T_NC=310  fn=1.063


In [2]:
transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,),(0.3081,))])
trainset  = torchvision.datasets.MNIST('/kaggle/working/data',
    train=True,  download=True, transform=transform)
testset   = torchvision.datasets.MNIST('/kaggle/working/data',
    train=False, download=True, transform=transform)
train_loader = DataLoader(trainset, batch_size=256, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'MNIST: {len(trainset):,} train / {len(testset):,} test')


100%|██████████| 9.91M/9.91M [00:00<00:00, 38.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.13MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.82MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.06MB/s]

MNIST: 60,000 train / 10,000 test


In [3]:
class MLP(nn.Module):
    def __init__(self, depth=5, width=512, act_cls=nn.ReLU, num_classes=10):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(784, width), act_cls()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), act_cls()]
        self.body   = nn.Sequential(*layers)
        self.head   = nn.Linear(width, num_classes)
        self._feats = None
        self.body.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.detach()))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def forward(self, x): return self.head(self.body(x))
    def get_features(self, x): self(x); return self._feats
    def get_classifier_weights(self): return self.head.weight.detach()

print('MLP defined.')


MLP defined.


In [4]:
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval()
    fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE)).cpu()); ll.append(y)
    H = torch.cat(fl).float(); Y = torch.cat(ll)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw   = sum((H[Y==c]-mu_c[c]).T@(H[Y==c]-mu_c[c]) for c in range(K))/len(H)
    Sb   = M.T @ M / K
    nc1  = (torch.trace(Sw)/torch.trace(Sb).clamp(1e-10)).item()
    Mn   = F.normalize(M, dim=1)
    cos  = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool)
    nc2  = (cos[mask]-(-1./(K-1))).abs().mean().item()
    Wn   = F.normalize(model.get_classifier_weights().cpu(), dim=1)
    nc3  = (1-(Mn*Wn).sum(1).mean()).item()
    return {'nc1':nc1,'nc2':nc2,'nc3':nc3,
            'feat_norm':H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(DEVICE),y.to(DEVICE)
            correct+=(model(x).argmax(1)==y).sum().item(); total+=len(y)
    return correct/total

def run(model, name, lr=1e-3, wd=1e-4,
        phase1=200, phase2=600, nc_every=10):
    # Always use NC1<0.01 threshold for wd sweep
    # so all data points are comparable (full collapse)
    nc_thresh = 0.01
    model=model.to(DEVICE); K=10; rows=[]; terminal=False; t0=time.time()
    for phase, loss_fn, n_ep in [(1,'ce',phase1),(2,'mse',phase2)]:
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep)
        off = phase1 if phase==2 else 0
        for ep_l in range(1, n_ep+1):
            ep = off + ep_l
            model.train()
            for x,y in train_loader:
                x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits=model(x)
                loss=(F.mse_loss(logits,F.one_hot(y,K).float())
                      if loss_fn=='mse' else F.cross_entropy(logits,y))
                loss.backward(); opt.step()
            sch.step()
            if ep_l%nc_every==0 or ep_l==n_ep:
                tr=evaluate(model,train_loader)
                te=evaluate(model,test_loader)
                if tr>=0.99 and not terminal:
                    terminal=True
                    print(f'  [{name}] Terminal ep={ep}')
                nc=(compute_nc(model,train_loader) if terminal
                    else {'nc1':None,'nc2':None,'nc3':None,'feat_norm':None})
                rows.append({'epoch':ep,'phase':phase,'train':tr,'test':te,**nc})
                nc1s=(f"{nc['nc1']:.5f}" if nc['nc1'] is not None else 'N/A')
                fns =(f"{nc['feat_norm']:.3f}" if nc['feat_norm'] else 'N/A')
                print(f'  ep={ep:>4} tr={tr:.4f} nc1={nc1s} '
                      f'fn={fns} t={(time.time()-t0)/60:.1f}m')
                if nc['nc1'] is not None and nc['nc1']<nc_thresh:
                    fn_val=nc['feat_norm']
                    print(f'  *** T_NC={ep}  fn={fn_val:.4f}  '
                          f'wd={wd}')
                    return pd.DataFrame(rows), ep, fn_val
    return pd.DataFrame(rows), None, None

print('run() ready. Using NC1<0.01 threshold for all wd runs.')


run() ready. Using NC1<0.01 threshold for all wd runs.


In [5]:
# Weight decay sweep: 4 values x 3 seeds = 12 runs
# wd=1e-4 baseline already done (T_NC=310, fn=1.063)
#
# KEY QUESTION: as wd changes, T_NC changes — but does fn at T_NC stay ~1.06?
# If yes: weight decay controls SPEED of collapse, not the THRESHOLD
# This is the direct analogue of your grokking H4 finding
#
# Expected behaviour based on grokking paper:
#   wd=1e-5: very slow (needs more phase2 epochs) — may DNF in 600ep
#   wd=5e-5: slower than baseline
#   wd=1e-4: baseline (T_NC=310, fn=1.063)
#   wd=5e-4: faster — stronger regularisation drives collapse sooner

WD_VALUES = [1e-5, 5e-5, 5e-4]   # 1e-4 already done as baseline
wd_results = []

for wd in WD_VALUES:
    for seed in range(3):
        name = f'wd{wd}_s{seed}'
        print(f'\n=== wd={wd}  seed={seed} ===')
        torch.manual_seed(seed)
        model = MLP(depth=5, width=512, act_cls=nn.ReLU)
        df, t_nc, fn = run(model, name, lr=1e-3, wd=wd,
                           phase1=200, phase2=600)
        fname = f'wd{str(wd).replace(".","p")}_s{seed}.csv'
        df.to_csv(SAVE_DIR + fname, index=False)
        wd_results.append({'wd':wd,'seed':seed,'T_NC':t_nc,
                           'fn':fn,'test_acc':df.test.iloc[-1]})
        status = f'T_NC={t_nc}  fn={fn:.4f}' if t_nc else 'DNF'
        print(f'  => {status}')

df_wd = pd.DataFrame(wd_results)
df_wd.to_csv(SAVE_DIR + 'sweep_wd.csv', index=False)

print('\n=== WEIGHT DECAY SWEEP SUMMARY ===')
print(df_wd.to_string())
print()
g = df_wd.dropna(subset=['fn']).groupby('wd')[['T_NC','fn']].agg(['mean','std'])
print('Group summary (confirmed T_NC only):')
print(g.to_string())



=== wd=1e-05  seed=0 ===
  [wd1e-05_s0] Terminal ep=10
  ep=  10 tr=0.9950 nc1=0.29633 fn=33.729 t=1.3m
  ep=  20 tr=0.9961 nc1=0.23619 fn=42.211 t=2.5m
  ep=  30 tr=0.9976 nc1=0.20553 fn=53.880 t=3.8m
  ep=  40 tr=0.9980 nc1=0.19336 fn=59.471 t=5.0m
  ep=  50 tr=0.9990 nc1=0.19316 fn=60.042 t=6.3m
  ep=  60 tr=0.9975 nc1=0.19574 fn=58.529 t=7.5m
  ep=  70 tr=0.9995 nc1=0.16591 fn=61.387 t=8.7m
  ep=  80 tr=0.9992 nc1=0.18634 fn=59.041 t=10.0m
  ep=  90 tr=0.9998 nc1=0.15211 fn=60.505 t=11.2m
  ep= 100 tr=0.9999 nc1=0.16414 fn=66.319 t=12.5m
  ep= 110 tr=1.0000 nc1=0.14855 fn=63.594 t=13.7m
  ep= 120 tr=0.9997 nc1=0.14824 fn=40.896 t=14.9m
  ep= 130 tr=1.0000 nc1=0.12277 fn=50.801 t=16.2m
  ep= 140 tr=1.0000 nc1=0.10342 fn=47.353 t=17.4m
  ep= 150 tr=1.0000 nc1=0.10894 fn=42.414 t=18.7m
  ep= 160 tr=1.0000 nc1=0.10359 fn=43.285 t=19.9m
  ep= 170 tr=1.0000 nc1=0.10565 fn=42.070 t=21.2m
  ep= 180 tr=1.0000 nc1=0.11124 fn=41.082 t=22.5m
  ep= 190 tr=1.0000 nc1=0.11565 fn=39.917 t=23.7m
 

In [6]:
# Include baseline in analysis
df_with_base = df_wd.copy()
baseline_rows = pd.DataFrame([
    {'wd':1e-4,'seed':0,'T_NC':310,'fn':1.063,'test_acc':0.9817}
])
df_all_wd = pd.concat([df_with_base, baseline_rows], ignore_index=True)

confirmed = df_all_wd.dropna(subset=['fn'])
fns  = confirmed.fn.values
tncs = confirmed.T_NC.values

print(f'Confirmed T_NC: N={len(fns)}')
print(f'fn values: {np.round(fns,4).tolist()}')
mean_fn = np.mean(fns); std_fn = np.std(fns); cv = std_fn/mean_fn
print(f'Mean fn: {mean_fn:.4f}  Std: {std_fn:.4f}  CV: {cv:.3f}')
if cv < 0.15:
    print('CV < 15% => CONSISTENT THRESHOLD across weight decay values!')
    print('Weight decay controls SPEED of collapse, not the threshold.')
else:
    print(f'CV={cv:.3f} — threshold varies with weight decay')

plt.rcParams.update({'font.family':'serif','font.size':11,
    'axes.spines.top':False,'axes.spines.right':False})
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) T_NC vs weight decay
ax = axes[0]
g = confirmed.groupby('wd')['T_NC'].agg(['mean','std'])
ax.errorbar(g.index, g['mean'], yerr=g['std'].fillna(0),
            fmt='o-', color='#2196F3', lw=2, ms=8, capsize=5)
ax.set_xscale('log')
ax.set(xlabel='Weight decay λ', ylabel='T_NC (epoch)',
       title='(a) Lambda controls collapse speed')
ax.grid(alpha=0.3)

# (b) feat_norm at T_NC vs weight decay  — the key panel
ax = axes[1]
g2 = confirmed.groupby('wd')['fn'].agg(['mean','std'])
ax.errorbar(g2.index, g2['mean'], yerr=g2['std'].fillna(0),
            fmt='s-', color='#E91E63', lw=2, ms=8, capsize=5)
ax.axhline(mean_fn, color='black', ls='--', lw=1.5,
           label=f'Grand mean={mean_fn:.3f}')
ax.set_xscale('log')
ax.set(xlabel='Weight decay λ', ylabel='Feature norm at T_NC',
       title=f'(b) Threshold vs lambda\nCV={cv:.3f}')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# (c) All fn values sorted
ax = axes[2]
sorted_fns = sorted(fns)
wd_labels = [f'wd={confirmed.sort_values("fn").wd.iloc[i]:.0e}'
             for i in range(len(sorted_fns))]
colors_wd = plt.cm.viridis(np.linspace(0.2,0.8,len(sorted_fns)))
ax.scatter(range(len(sorted_fns)), sorted_fns,
           c=colors_wd, s=80, zorder=3)
ax.axhline(mean_fn, color='black', ls='--', lw=2,
           label=f'Mean={mean_fn:.3f}')
ax.fill_between(range(len(sorted_fns)),
                mean_fn-std_fn, mean_fn+std_fn,
                alpha=0.12, color='black',
                label=f'±1σ  CV={cv:.3f}')
ax.set(xlabel='Run (sorted by fn)',
       ylabel='Feature norm at T_NC',
       title='(c) All confirmed T_NC values')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

fig.suptitle('Weight Decay Sweep: NC1 Collapse Dynamics | '
             'MLP-5 | ReLU | MNIST | Two-phase',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig_wd_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_wd_sweep.png')
print('Saved: sweep_wd.csv')


Confirmed T_NC: N=7
fn values: [0.9544, 1.1264, 1.0236, 0.9761, 0.9847, 0.9879, 1.063]
Mean fn: 1.0166  Std: 0.0555  CV: 0.055
CV < 15% => CONSISTENT THRESHOLD across weight decay values!
Weight decay controls SPEED of collapse, not the threshold.
Saved: fig_wd_sweep.png
Saved: sweep_wd.csv
